# Train CatBoost with engineered features
Every feature is computed from the passenger's **own row** (or their travel group, which never spans
train and test), so nothing is learned from other rows' labels or statistics, and nothing leaks.

Data split: train : validation : test = 60 : 20 : 20.
- **5-fold CV** on train+validation (80%) is the model-selection score and picks the number of trees.
- The **test** 20% is scored once at the end as an honest estimate of the Kaggle score.

In [1]:
import numpy as np
import pandas as pd

target = "transported"
MONEY_COLS = ["roomservice", "foodcourt", "shoppingmall", "spa", "vrdeck"]


def standardize_columns(df):
    """Strip, lowercase and snake_case column names (same as visual.ipynb)."""
    out = df.copy()
    out.columns = out.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
    return out


df = standardize_columns(pd.read_csv("./data/raw/train.csv"))
df.head()

,passengerid,homeplanet,cryosleep,cabin,destination,age,vip,roomservice,foodcourt,shoppingmall,spa,vrdeck,name,transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## Feature engineering (row-level, no fitted statistics)
| Feature | Built from |
|---|---|
| `group_size`, `solo` | group id = first 4 digits of `passengerid` (`0003_01` → `0003`) |
| `deck`, `cabin_num`, `side` | `cabin` = `deck/num/side`, e.g. `B/0/P` |
| logic fills | people in cryosleep can't spend → missing spend = 0; anyone who spent wasn't asleep |
| `total_spend`, `no_spend`, `log_*` | the five spending columns |

Missing categories become their own `"missing"` value instead of a fitted mode.

In [2]:
def engineer(df):
    """Add features using only each row (and its own travel group). Never drops rows."""
    out = df.copy()

    group = out["passengerid"].str.split("_").str[0]
    out["group_size"] = group.map(group.value_counts())
    out["solo"] = (out["group_size"] == 1).astype(int)

    cabin = out["cabin"].str.split("/", expand=True)
    out["deck"] = cabin[0]
    out["cabin_num"] = pd.to_numeric(cabin[1])
    out["side"] = cabin[2]

    asleep = out["cryosleep"] == True  # noqa: E712  (column holds True/False/NaN)
    out.loc[asleep, MONEY_COLS] = out.loc[asleep, MONEY_COLS].fillna(0)
    spent = out[MONEY_COLS].sum(axis=1) > 0
    out.loc[out["cryosleep"].isna() & spent, "cryosleep"] = False

    out["total_spend"] = out[MONEY_COLS].sum(axis=1)
    out["no_spend"] = (out["total_spend"] == 0).astype(int)
    for col in MONEY_COLS + ["total_spend"]:
        out[f"log_{col}"] = np.log1p(out[col])
    return out


CAT_FEATURES = ["homeplanet", "cryosleep", "destination", "vip", "deck", "side"]
NUM_FEATURES = (["age", "group_size", "solo", "cabin_num", "total_spend", "no_spend"]
                + [f"log_{c}" for c in MONEY_COLS + ["total_spend"]])
selected_features = CAT_FEATURES + NUM_FEATURES
selected_cat_features = CAT_FEATURES  # name kept for evaluate-model.ipynb / predict.ipynb


def to_model_input(df):
    """Pick features; CatBoost wants categorical columns as strings (missing → "missing")."""
    X = df[selected_features].copy()
    for col in CAT_FEATURES:
        X[col] = X[col].map(lambda v: "missing" if pd.isna(v) else str(v))
    return X


df_fe = engineer(df)
print(len(selected_features), "features:", selected_features)
df_fe[selected_features].head()

18 features: ['homeplanet', 'cryosleep', 'destination', 'vip', 'deck', 'side', 'age', 'group_size', 'solo', 'cabin_num', 'total_spend', 'no_spend', 'log_roomservice', 'log_foodcourt', 'log_shoppingmall', 'log_spa', 'log_vrdeck', 'log_total_spend']


,homeplanet,cryosleep,destination,vip,deck,side,age,group_size,solo,cabin_num,total_spend,no_spend,log_roomservice,log_foodcourt,log_shoppingmall,log_spa,log_vrdeck,log_total_spend
0,Europa,False,TRAPPIST-1e,False,B,P,39.0,1,1,0.0,0.0,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,Earth,False,TRAPPIST-1e,False,F,S,24.0,1,1,0.0,736.0,0,4.700480,2.302585,3.258097,6.309918,3.806662,6.602588
2,Europa,False,TRAPPIST-1e,True,A,S,58.0,2,0,0.0,10383.0,0,3.784190,8.182280,0.000000,8.812248,3.912023,9.248021
3,Europa,False,TRAPPIST-1e,False,A,S,33.0,2,0,0.0,5176.0,0,0.000000,7.157735,5.918894,8.110728,5.267858,8.551981
4,Earth,False,TRAPPIST-1e,False,F,S,16.0,1,1,1.0,1091.0,0,5.717028,4.262680,5.023881,6.338594,1.098612,6.995766


## Split: train : validation : test = 60 : 20 : 20

In [3]:
from sklearn.model_selection import train_test_split

train_val_df, test_df = train_test_split(df_fe, test_size=0.2, random_state=42,
                                         stratify=df_fe[target])
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42,
                                    stratify=train_val_df[target])

X_train_cb, X_val_cb, X_test_cb = (to_model_input(d) for d in (train_df, val_df, test_df))
y_train, y_val, y_test = (d[target].astype(int) for d in (train_df, val_df, test_df))

# this is use for prediction
print(X_train_cb.shape, X_val_cb.shape, X_test_cb.shape)

# this is use for answer
print(y_train.shape, y_val.shape, y_test.shape)

(5215, 18) (1739, 18) (1739, 18)
(5215,) (1739,) (1739,)


## 5-fold cross-validation on train+validation
More reliable than one validation split. Each fold early-stops on Logloss; the median best iteration
becomes the tree count for the final model in `predict.ipynb`.

In [4]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

CATBOOST_PARAMS = dict(learning_rate=0.03, depth=6, l2_leaf_reg=3,
                       random_seed=42, allow_writing_files=False)

X_cv = to_model_input(train_val_df)
y_cv = train_val_df[target].astype(int)

cv_acc, cv_best_iters = [], []
for fold, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(X_cv, y_cv)):
    m = CatBoostClassifier(**CATBOOST_PARAMS, iterations=3000, early_stopping_rounds=200, verbose=0)
    m.fit(X_cv.iloc[tr], y_cv.iloc[tr], cat_features=CAT_FEATURES,
          eval_set=(X_cv.iloc[va], y_cv.iloc[va]))
    acc = accuracy_score(y_cv.iloc[va], m.predict(X_cv.iloc[va]).astype(int))
    cv_acc.append(acc)
    cv_best_iters.append(m.get_best_iteration() + 1)
    print(f"fold {fold}: acc={acc:.4f}  trees={cv_best_iters[-1]}")

n_trees = int(np.median(cv_best_iters))
print(f"CV accuracy = {np.mean(cv_acc):.4f} ± {np.std(cv_acc):.4f}  | trees for final model: {n_trees}")

fold 0: acc=0.8109  trees=973


fold 1: acc=0.8167  trees=901


fold 2: acc=0.8160  trees=1021


fold 3: acc=0.8095  trees=1138


fold 4: acc=0.8115  trees=806
CV accuracy = 0.8129 ± 0.0029  | trees for final model: 973


## Train CatBoost on the train split
Early stopping on the validation split. This `model` is what `evaluate-model.ipynb` inspects.

In [5]:
model = CatBoostClassifier(**CATBOOST_PARAMS, iterations=3000, eval_metric="Accuracy",
                           custom_metric=["AUC"], early_stopping_rounds=200, verbose=200)
model.fit(X_train_cb, y_train,
          cat_features=CAT_FEATURES,
          eval_set=(X_val_cb, y_val))

0:	learn: 0.7626079	test: 0.7492812	best: 0.7492812 (0)	total: 40.2ms	remaining: 2m


200:	learn: 0.8214765	test: 0.7987349	best: 0.8010351 (185)	total: 9.79s	remaining: 2m 16s


400:	learn: 0.8448706	test: 0.8044853	best: 0.8056354 (393)	total: 19.5s	remaining: 2m 6s


600:	learn: 0.8671141	test: 0.8056354	best: 0.8073606 (543)	total: 29.5s	remaining: 1m 57s


Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.807360552
bestIteration = 543

Shrink model to first 544 iterations.


CatBoostClassifier(allow_writing_files=False, custom_metric=['AUC'], depth=6, early_stopping_rounds=200, eval_metric='Accuracy', iterations=3000, l2_leaf_reg=3, learning_rate=0.03, random_seed=42, verbose=200)

In [6]:
def report(name, X, y):
    pred = model.predict(X).astype(int).ravel()
    proba = model.predict_proba(X)[:, 1]
    print(f"{name:10s} acc={accuracy_score(y, pred):.4f}  "
          f"f1={f1_score(y, pred):.4f}  auc={roc_auc_score(y, proba):.4f}")

report("train", X_train_cb, y_train)
report("validation", X_val_cb, y_val)

train      acc=0.8531  f1=0.8552  auc=0.9416
validation acc=0.8074  f1=0.8078  auc=0.8964


In [7]:
model.get_feature_importance(prettified=True)

,Feature Id,Importances
0,homeplanet,12.198005
1,log_spa,11.335965
2,cabin_num,10.696575
3,deck,8.373850
4,log_vrdeck,7.961254
5,cryosleep,7.711922
6,log_foodcourt,7.197546
7,log_total_spend,5.986637
8,log_roomservice,5.503172
9,side,5.420054


## Final evaluation on test (run once)

In [8]:
report("test", X_test_cb, y_test)

test       acc=0.8051  f1=0.8066  auc=0.9099
